<div style='background:linear-gradient(135deg,#1B3A6B,#2D5AA0);padding:30px;border-radius:12px;color:white;font-family:Arial'>
<h1 style='margin:0;font-size:28px'>🚀 Chapter 17 — Model Deployment and Production</h1>
<p style='margin:8px 0 0'>Book: Machine Learning — From Zero to Engineer | Est. Time: 60 mins | Level: Advanced</p>
</div>

## 📋 Learning Objectives

By the end of this notebook, you will be able to:
- Save and load ML models using pickle and joblib
- Version models systematically
- Simulate a REST API prediction endpoint in Python
- Implement input validation for model serving
- Monitor model performance drift over time

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# Setup & Imports
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import joblib
import json
import os
import datetime

from sklearn.ensemble        import GradientBoostingClassifier
from sklearn.preprocessing   import StandardScaler
from sklearn.pipeline        import Pipeline
from sklearn.datasets        import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics         import accuracy_score, roc_auc_score

%matplotlib inline
np.random.seed(42)

# Synthetic data
X, y = make_classification(n_samples=1000, n_features=8, n_informative=6, random_state=42)
FEATURE_NAMES = [f'feature_{i}' for i in range(8)]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 📖 Section A — Saving and Loading Models

```python
import joblib, pickle

# Save with joblib (recommended for sklearn — handles numpy arrays efficiently)
joblib.dump(pipeline, 'model_v1.0.joblib')
pipeline = joblib.load('model_v1.0.joblib')

# Save with pickle (universal Python serialisation)
with open('model.pkl', 'wb') as f:
    pickle.dump(pipeline, f)
with open('model.pkl', 'rb') as f:
    pipeline = pickle.load(f)
```

> ⚠️ **Security Warning:** Never load pickle/joblib files from untrusted sources — they can execute arbitrary code. For production, use ONNX or model registries (MLflow, SageMaker) which have safer serialisation.

> 💡 **Key Rule:** Always save the FULL PIPELINE (preprocessor + model together), not just the model. If you save only the model, you must manually apply the same preprocessing at inference — prone to bugs and training-serving skew.

---
## 🟢 Exercise 17.1 — Save, Load, and Verify *(Level 1 · Est. 10 min)*

> Train a GBM pipeline, save with joblib, reload from disk, verify predictions match.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 17.1: Save and Load Model
# ─────────────────────────────────────────────────────────────

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  GradientBoostingClassifier(n_estimators=50, random_state=42))
])
pipeline.fit(X_train, y_train)

# Save the pipeline
model_path = 'model_v1.0.joblib'
# YOUR CODE HERE

# Load the pipeline
# YOUR CODE HERE

# Verify predictions match
pred_original = pipeline.predict(X_test)
pred_loaded   = # YOUR CODE HERE

assert np.array_equal(pred_original, pred_loaded), 'Predictions must match!'
print(f'Model saved to: {model_path}')
print(f'File size: {os.path.getsize(model_path)/1024:.1f} KB')
print(f'Accuracy: {accuracy_score(y_test, pred_loaded):.4f}')
print('✅ Save/load verified!')

In [ ]:
# @title ✅ Solution — Exercise 17.1 (click to expand)

model_path = 'model_v1.0.joblib'
joblib.dump(pipeline, model_path)

loaded_pipeline = joblib.load(model_path)

pred_original = pipeline.predict(X_test)
pred_loaded   = loaded_pipeline.predict(X_test)

assert np.array_equal(pred_original, pred_loaded)
print(f'Model saved to: {model_path}')
print(f'File size: {os.path.getsize(model_path)/1024:.1f} KB')
print(f'Test accuracy: {accuracy_score(y_test, pred_loaded):.4f}')
print('✅ Loaded model produces identical predictions.')

---
## 🟡 Exercise 17.2 — Simulate a Prediction API *(Level 2 · Est. 20 min)*

> Write a `predict_api()` function that takes a JSON payload, validates inputs, preprocesses, and returns a structured JSON response with prediction and confidence.

In [ ]:
# @title ✅ Solution — Exercise 17.2 (click to expand)

EXPECTED_FEATURES = FEATURE_NAMES
FEATURE_RANGES = {f: (-5.0, 5.0) for f in FEATURE_NAMES}  # expected range per feature

def validate_input(payload: dict) -> tuple:
    """
    Validate incoming JSON payload.
    Returns (is_valid: bool, error_message: str)
    """
    for feat in EXPECTED_FEATURES:
        if feat not in payload:
            return False, f'Missing feature: {feat}'
        val = payload[feat]
        if not isinstance(val, (int, float)):
            return False, f'Feature {feat} must be numeric, got {type(val).__name__}'
    return True, None

def predict_api(payload: dict, model) -> dict:
    """
    Simulate a REST API prediction endpoint.
    Input: JSON dict with feature values
    Output: JSON dict with prediction and confidence
    """
    is_valid, error = validate_input(payload)
    if not is_valid:
        return {'status': 'error', 'message': error}

    X_input = np.array([[payload[f] for f in EXPECTED_FEATURES]])
    prediction  = int(model.predict(X_input)[0])
    confidence  = float(model.predict_proba(X_input)[0][prediction])

    return {
        'status':     'success',
        'prediction': prediction,
        'confidence': round(confidence, 4),
        'label':      'default' if prediction == 1 else 'no_default',
        'timestamp':  datetime.datetime.utcnow().isoformat() + 'Z',
        'model_version': 'v1.0'
    }

# Test valid input
sample_payload = {f: float(X_test[0, i]) for i, f in enumerate(FEATURE_NAMES)}
result = predict_api(sample_payload, loaded_pipeline)
print('Valid input response:')
print(json.dumps(result, indent=2))

# Test invalid input
bad_payload = {'feature_0': 1.5}  # missing most features
result_bad = predict_api(bad_payload, loaded_pipeline)
print('\nInvalid input response:')
print(json.dumps(result_bad, indent=2))

---
## 🔴 Exercise 17.3 — Model Performance Monitoring *(Level 3 · Est. 20 min)*

> Simulate 12 months of production data where model performance drifts over time. Plot rolling accuracy. Detect the month when drift exceeds a threshold.

In [ ]:
# @title ✅ Solution — Exercise 17.3 (click to expand)

np.random.seed(99)
monthly_accs = []
DRIFT_THRESHOLD = 0.05  # flag if accuracy drops more than 5% from baseline

baseline_acc = accuracy_score(y_test, loaded_pipeline.predict(X_test))

for month in range(1, 13):
    # Simulate data drift: features shift gradually
    drift_factor = 1 + (month / 12) * 0.5  # increasing drift
    X_drift, y_drift = make_classification(
        n_samples=200, n_features=8, n_informative=6,
        random_state=month * 7, class_sep=1.0 / drift_factor
    )
    preds = loaded_pipeline.predict(X_drift)
    acc   = accuracy_score(y_drift, preds)
    monthly_accs.append({'month': month, 'accuracy': acc})

monitor_df = pd.DataFrame(monthly_accs)

plt.figure(figsize=(10, 5))
plt.plot(monitor_df['month'], monitor_df['accuracy'], 'bo-', label='Monthly Accuracy')
plt.axhline(baseline_acc,                    color='green', linestyle='--', label=f'Baseline ({baseline_acc:.3f})')
plt.axhline(baseline_acc - DRIFT_THRESHOLD,  color='red',   linestyle='--', label=f'Drift Threshold')
plt.fill_between(monitor_df['month'],
                 baseline_acc - DRIFT_THRESHOLD, monitor_df['accuracy'],
                 where=(monitor_df['accuracy'] < baseline_acc - DRIFT_THRESHOLD),
                 color='red', alpha=0.2, label='Drift detected')
plt.title('Model Performance Monitoring — 12 Month Simulation')
plt.xlabel('Month')
plt.ylabel('Accuracy')
plt.legend()
plt.xticks(range(1, 13))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

drift_months = monitor_df[monitor_df['accuracy'] < baseline_acc - DRIFT_THRESHOLD]
if len(drift_months):
    print(f'Drift detected at month(s): {drift_months["month"].tolist()}')
    print('Action: Retrain model on recent data!')
else:
    print('No significant drift detected.')

---
## 🎤 Interview Q&A

<details>
<summary><strong>Q1: What is training-serving skew?</strong></summary>

**Answer:** Training-serving skew occurs when the preprocessing applied at training time differs from the preprocessing applied at prediction time. Example: you fit the scaler on training data during experimentation, but forget to apply the same scaling at inference — predictions are garbage. Prevention: always save and load the FULL pipeline, never just the model. Use feature stores to ensure consistent feature computation between training and serving.
</details>

<details>
<summary><strong>Q2: What is model drift and how do you detect it?</strong></summary>

**Answer:** Data drift: input feature distributions change over time (customers' spending patterns shift post-COVID). Concept drift: the relationship between features and target changes (fraudsters adapt to the model). Detection: (1) Monitor feature statistics (mean, std, % missing) vs training baseline. (2) Monitor prediction distributions (if 90% of predictions are now class 1, something changed). (3) Monitor actual performance metrics if labels are available. Alert when metrics cross a threshold, then retrain.
</details>

<details>
<summary><strong>Q3: How does a model get from a notebook to a REST API?</strong></summary>

**Answer:** Typical path: (1) Train and evaluate in Jupyter notebook. (2) Save model with joblib/MLflow. (3) Write a prediction function with input validation. (4) Wrap in a Flask/FastAPI web server — one `/predict` endpoint. (5) Containerise with Docker. (6) Deploy to cloud (AWS SageMaker, GCP Vertex AI, Azure ML, or Kubernetes). (7) Set up monitoring with CloudWatch/Grafana. In India, common stack: FastAPI + Docker + AWS EC2 or Azure App Service.
</details>

---

<div style='background:#EDF7F0;border-left:6px solid #2E7D50;padding:20px;border-radius:8px;margin-top:20px'>
<h3 style='color:#2E7D50'>✅ Chapter 17 Complete!</h3>
<ul>
<li>Model serialisation with joblib and pickle</li>
<li>Input validation for production APIs</li>
<li>Simulated REST prediction endpoint</li>
<li>Model performance monitoring and drift detection</li>
</ul>
<p><strong>Next:</strong> Chapter 18 — Ethics and Responsible AI</p>
</div>